# FPGA Camera Setup

This notebook documents the baseline OV7670 camera setup used with the ZedBoard before any dataset-specific image preprocessing or neural-network integration was added.

The goal of this stage was to establish a reusable hardware path in which the camera could be configured, camera pixels could be captured into FPGA memory, and the resulting image could be displayed through VGA.

The same camera baseline was used as the starting point for the later dataset-specific FPGA implementations.

## Scope

This notebook covers:

- physical OV7670-to-ZedBoard wiring;
- camera configuration and supporting VHDL;
- camera pixel capture;
- frame storage in Block RAM;
- VGA output;
- Vivado project dependencies; and
- functional validation of the camera feed.

This notebook does **not** cover:

- MNIST-specific ROI selection;
- 28×28 downsampling;
- thresholding or binarization;
- neural-network input preparation;
- zyNet integration;
- inference control;
- performance benchmarking; or
- cloud-assisted FPGA processing.

## 1. Hardware and software

The baseline camera setup used:

- **ZedBoard Rev E**
- **Xilinx Zynq-7000 XC7Z020CLG484-1**
- **OV7670 camera module**
- **VGA display**
- **Vivado 2018.1**
- jumper wires for the camera-to-board connection

The camera communicated with the programmable logic through its parallel pixel interface and control signals. The design configured the camera through its serial control interface, captured incoming pixel data, stored the frame in Block RAM, and read the stored image for VGA display.

## 2. Source design and attribution

The baseline implementation was adapted from:

**Alex Zhang, OV7670 Guide**  
https://github.com/Alexzhang1988/OV7670_guide

The source project itself is based primarily on earlier OV7670 FPGA work by **Mike Field**. Original attribution comments are preserved in several of the VHDL files.

The camera register configuration also includes settings derived in part from the Linux kernel `OV7670.c` driver, as identified in the original source comments.

The **OmniVision OV7670/OV7171 Implementation Guide** was used as supporting documentation for the camera interface, synchronization signals, register behavior, and SCCB configuration.

The baseline project was adapted for the ZedBoard and used as the common camera foundation for the later FPGA experiments.

## 3. Baseline source files

The camera-only code branch contains the files required for the baseline implementation.

| File | Purpose |
|---|---|
| `ov7670_top.vhd` | Camera-only top-level design |
| `ov7670_controller.vhd` | Camera initialization control |
| `ov7670_registers.vhd` | OV7670 register configuration |
| `ov7670_capture.vhd` | Camera pixel capture and frame-memory write logic |
| `i2c_sender.vhd` | Serial camera-control communication |
| `vga.vhd` | VGA timing and frame-memory display logic |
| `debounce.vhd` | Button/switch debounce support |
| `clocking.vhd` | Clock-generation logic |
| `ZedBoard.xdc` | ZedBoard physical pin constraints |
| `blk_mem_gen_0.xci` | Vivado Block Memory Generator configuration |

Generated synthesis, implementation, cache, and run directories are not required because Vivado can regenerate them from the source files and IP configuration.

The `ZedBoard.xdc` file should be treated as the authoritative source for the FPGA package-pin assignments.

## 4. OV7670 interface signals

The OV7670 interface uses the following signal groups:

### Camera control

- `SIOC` / serial clock
- `SIOD` / serial data
- `RESET`
- `PWDN`
- camera input clock (`XCLK`)

### Camera timing

- `PCLK`
- `VSYNC`
- `HREF`

### Pixel data

- `D0` through `D7`

The OmniVision documentation describes `VSYNC` as the vertical synchronization signal, `HREF` as the horizontal data-valid reference, and `PCLK` as the pixel clock used to capture the parallel camera data.

Power and ground were supplied separately through the appropriate camera and ZedBoard header connections.

## 5. Physical wiring

The camera was connected to the ZedBoard using individual jumper wires routed to the board headers defined in the constraints file.

The photographs below document the physical wiring used during development.

### Early header wiring

![Close-up of ZedBoard camera header wiring](images/camera_zedboard_header_wiring.jpg)

### ZedBoard wiring overview

![ZedBoard with camera jumper wiring](images/camera_zedboard_overview.jpg)

### OV7670 connection overview

![OV7670 camera connected to the ZedBoard](images/ov7670_wiring_overview.jpg)

### OV7670 pin labels

![OV7670 module showing labeled pins](images/ov7670_pin_labels.jpg)

## 6. Wiring notes and final placement

The handwritten wiring notes were created while the camera connections were being established and debugged.

![Handwritten camera wiring notes](images/camera_wiring_notes.jpg)

Because jumper colors and intermediate placements could change during troubleshooting, the handwritten notes and earlier photographs should be treated as historical setup documentation rather than the final authority for signal assignment.

The final physical placement is shown below.

![Final OV7670-to-ZedBoard wiring placement](images/final_camera_wiring_placement.jpeg)

For reproduction:

1. use `ZedBoard.xdc` to determine the actual FPGA pin assignments;
2. use the final wiring photograph as a physical routing reference; and
3. use the OV7670 module labels to verify each camera-side signal before powering the board.

## 7. Camera data path

The baseline camera design implements the following path:

```text
OV7670 Camera
      |
      v
Camera Configuration
      |
      v
Pixel Capture
      |
      v
Block RAM Frame Buffer
      |
      v
VGA Display
```

The camera controller programs the OV7670 registers. During active video, the capture logic receives the parallel camera data together with the camera timing signals and writes the resulting frame data to Block RAM.

The VGA logic reads the stored frame and generates the display timing required to show the captured image on an external monitor.

This stage establishes the camera and display infrastructure only. Dataset-specific processing is added later.

## 8. Vivado Block RAM

The baseline design uses a Vivado Block Memory Generator for frame storage.

The repository includes:

```text
blk_mem_gen_0.xci
```

This file preserves the IP configuration needed to recreate the frame-memory block in Vivado.

The camera capture logic writes image data to the frame buffer, while the VGA path reads the stored image for display.

No dataset-specific processed-image memory is part of this baseline notebook.

## 9. Functional validation and troubleshooting

Camera validation was performed incrementally. The photographs document several stages of the process rather than only the final working state.

### Initial corrupted VGA output

![Initial striped/corrupted VGA camera output](images/vga_initial_corrupted_output.jpg)

The display produced structured image data, but the frame was not yet being reconstructed correctly.

### Partial valid camera display

![Small valid camera image during intermediate testing](images/vga_partial_camera_output.jpg)

A valid image region was visible, indicating that camera configuration, pixel capture, frame storage, and VGA output were beginning to operate together.

### Working VGA camera feed

![Working OV7670 camera feed displayed through VGA](images/vga_working_camera_output.jpg)

The final baseline configuration displayed a recognizable live camera image through VGA. The printed digit visible in the photograph was simply an object used to make the camera output easy to recognize; this stage did not yet perform dataset-specific preprocessing or neural-network inference.

## 10. Baseline validation criteria

The camera setup was considered ready for later FPGA experiments after the following conditions were met:

- the OV7670 could be initialized successfully;
- camera timing signals were accepted by the FPGA capture logic;
- pixel data could be written to the frame buffer;
- the stored frame could be read by the VGA logic;
- the VGA monitor displayed a recognizable camera image; and
- the camera operated through the ZedBoard programmable logic without requiring neural-network hardware.

This baseline separated camera-interface problems from later image-processing and NN-integration problems.

## 11. Relationship to later dataset implementations

This notebook intentionally ends at the working camera feed.

Later dataset-specific FPGA branches extend this baseline with additional processing such as:

```text
OV7670 Camera
      |
      v
Baseline Camera Capture
      |
      v
Dataset-Specific Preprocessing
      |
      v
Processed Image Storage
      |
      v
Neural-Network Inference
```

Keeping the camera setup separate allows the same physical interface and baseline capture path to be reused across MNIST, EMNIST, SVHN, CIFAR, and other FPGA experiments.

## 12. Reproduction checklist

To recreate the baseline camera setup:

1. Create a Vivado 2018.1 project targeting the ZedBoard XC7Z020CLG484-1.
2. Add the baseline camera VHDL files.
3. Add `ZedBoard.xdc`.
4. Add `blk_mem_gen_0.xci` or recreate the equivalent Block Memory Generator.
5. Add the clocking source/IP used by the project.
6. Connect the OV7670 according to the constraints file and verify the camera-side labels.
7. Connect the ZedBoard VGA output to a display.
8. Synthesize and implement the design.
9. Program the ZedBoard.
10. Confirm that a recognizable live camera image appears on the VGA display.

If the displayed image is corrupted, verify the camera wiring, clocking, synchronization signals, frame-memory configuration, and VGA timing before adding any dataset-specific logic.

## Repository image files

The notebook expects the following images in an `images/` directory:

```text
camera_zedboard_header_wiring.jpg
camera_zedboard_overview.jpg
ov7670_wiring_overview.jpg
ov7670_pin_labels.jpg
camera_wiring_notes.jpg
final_camera_wiring_placement.jpeg
vga_initial_corrupted_output.jpg
vga_partial_camera_output.jpg
vga_working_camera_output.jpg
```

Additional setup photographs may remain in the repository even if they are not embedded directly in the notebook.